# Lab 10c: translating the Fed's 2026 severely adverse scenario

**AI-powered Investment and Risk Management (25882), Lecture 10: AI in Risk Forecasting and Stress Testing**

A supervisory scenario is a set of macroeconomic paths: unemployment, house prices, interest rates, stock prices. It says nothing directly about *your* losses. The step in between is **translation**: a model, often called a satellite model, that maps the scenario's variables into the returns or losses you care about. That is where machine learning is most often proposed, and where the extrapolation trap of the lecture lives.

This lab takes the Federal Reserve's own 2026 severely adverse scenario and translates it into the returns of US bank stocks with four models.

**You will**

- read the scenario against history and find where it leaves the data;
- fit four satellite models: linear regression, gradient boosting, gradient boosting with economic sign constraints, and a hybrid;
- backtest them on 2007–2009 and 2020 before trusting any of them;
- translate the 2026 scenarios and see how much the choice of model matters.

**Data.** The Fed's 2026 stress-test files (historical domestic variables and the final severely adverse and baseline scenarios, February 2026) and the Banks industry portfolio from the Kenneth R. French Data Library, all mirrored in the teaching repository. The satellite sample runs from 1989, when the Fed's BBB yield series begins.

**Runtime.** Well under a minute on a laptop.

## Setup

In [ ]:
import os
import urllib.request

if not os.path.exists("airm10.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/VitaliAlexeev/AI_Investments_2026/main/airm10.py", "airm10.py")

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

import airm10 as A

## 1. Read the scenario first

Six of the scenario's variables, from 2000 onwards, with the severely adverse and baseline paths attached at the end of 2025.

In [ ]:
hist = A.load_fed("history")
severe = A.load_fed("severely adverse")
base = A.load_fed("baseline")
A.scenario_panel_figure(hist, {"Severely adverse": severe, "Baseline": base}).show()
print("3-month Treasury rate in the severely adverse scenario:", severe["tb3"].tolist())

Three things stand out.

- **The size of the first quarter.** The stock market falls 38.7% in the first quarter of 2026 alone, and 57.8% by its lowest point. House prices fall 29.7% and commercial property 38.8%, and unemployment peaks at 10.0%.
- **Rates fall, again.** The 3-month Treasury rate is 2.5% in the first quarter and 0.1% in every one of the next 12 quarters. That is the template the lecture's problem slide described for 2015–2022, still in place in 2026.
- **It leaves history in the first quarter.** The table below lists every scenario quarter and variable outside the range of the 1989–2025 satellite sample.

In [ ]:
X_hist = A.fed_features(hist)[A.SATELLITE_FEATURES].dropna()
X_severe = A.fed_scenario_features("severely adverse")[A.SATELLITE_FEATURES]
X_base = A.fed_scenario_features("baseline")[A.SATELLITE_FEATURES]
display(A.beyond_range(X_hist, X_severe))

In the first quarter, four of the six variables are beyond anything since 1989: the stock market's −38.7% against a worst of −23.4% (2008Q4), a 3.4 percentage-point jump in the BBB spread against 3.0, and a −6.3% quarter for house prices against a worst of −5.2%. Any model that cannot extrapolate will have to guess there.

## 2. Four satellite models

The target is the quarterly return of US bank stocks. The inputs are the quarter's stock market return, the change in the BBB spread over the 10-year Treasury, the change in unemployment, house price growth, commercial property price growth and the change in the 10-year yield.

- **Linear regression.** One coefficient per variable; extrapolates.
- **Boosting.** Gradient-boosted trees, free to find any shape; flat beyond the data.
- **Monotone boosting.** The same, constrained to economically sensible signs: bank stocks must not *rise* when the market falls, spreads widen, unemployment rises or property prices fall. The 10-year yield is left free.
- **Hybrid.** The linear model plus boosting on its residuals, with the correction switched off whenever any input is outside the training range.

<details>
<summary><b>The maths: satellite models</b></summary>

A satellite model is a regression $y_t = f(x_t) + \varepsilon_t$ of a portfolio's return $y_t$ on the scenario variables $x_t$ in the same quarter. A scenario supplies a path $x_{T+1},\dots,x_{T+13}$, and the translated path is $\hat y_{T+h} = \hat f(x_{T+h})$, compounded into a level: $100\prod_{h}(1+\hat y_{T+h}/100)$.

A monotone constraint $+1$ on variable $j$ forces $\hat f$ to be non-decreasing in $x_j$, holding the others fixed; $-1$ forces it to be non-increasing; $0$ leaves it free. The constraints restrict the *direction* of the response, not its *level*, so a monotone tree ensemble is still flat beyond the training range.
</details>

In [ ]:
data = A.satellite_data(target="Banks")
FEATURES, MONOTONE = A.SATELLITE_FEATURES, A.MONOTONE
print(f"{len(data)} quarters, {data.index[0]} to {data.index[-1]}")

def fit_satellites(d):
    X, y = d[FEATURES].values, d["target"].values
    hgb = dict(max_iter=300, learning_rate=0.05, min_samples_leaf=10, random_state=0)
    return {"Linear": LinearRegression().fit(X, y),
            "Boosting": HistGradientBoostingRegressor(**hgb).fit(X, y),
            "Monotone boosting": HistGradientBoostingRegressor(monotonic_cst=MONOTONE, **hgb).fit(X, y),
            "Hybrid": A.HybridRegressor().fit(X, y)}

full = fit_satellites(data)
coefficients = pd.Series(full["Linear"].coef_, index=[A.FEATURE_LABELS[f] for f in FEATURES])
display(coefficients.round(2).to_frame("Linear coefficient"))

The linear model says bank stocks move about 1.03 for 1 with the market, and lose about 3.2 points more for every percentage point the BBB spread widens.

## 3. Backtest before you trust

A translation model can be checked where a scenario cannot: feed it the macro path that *actually* happened, and compare its answer with what bank stocks actually did. Each model is trained only on data before the episode.

In [ ]:
BACKTESTS = {"Global financial crisis": ("2006Q4", "2007Q1", "2009Q4"), "COVID": ("2019Q4", "2020Q1", "2020Q4")}
rows = []
for label, (train_end, start, end) in BACKTESTS.items():
    models = fit_satellites(data.loc[:train_end])
    test = data.loc[start:end]
    preds = {name: m.predict(test[FEATURES].values) for name, m in models.items()}
    A.path_figure(preds, test.index, actual=test["target"].values,
                  title=f"{label}: trained to {train_end}, given the macro path that actually happened").show()
    for name, p in {"Actual": test["target"].values, **preds}.items():
        level = A.cumulative(p)
        rows.append({"Episode": label, "Model": name, "Worst quarter (%)": p.min(),
                     "Lowest point (%)": level.min() - 100, "End of window (%)": level[-1] - 100})
backtests = pd.DataFrame(rows).set_index(["Episode", "Model"])
display(backtests.round(1))

**The global financial crisis.** Bank stocks' lowest point was −71.2%, and they ended 2009 at −55.7%. Given the actual macro path, the linear model's lowest point was only −21.7%, and it ended the window at +23.0%: it read the 2009 stock rally as good news for banks. Monotone boosting came closest, with a lowest point of −61.3%. The hybrid matches the linear model here because in most of these quarters at least one input was outside its training range, so its correction was switched off.

**COVID.** Banks lost 35.2% in the first quarter of 2020; the models predicted between −18.3% and −23.3%. None of them saw it.

The lesson cuts both ways. Inside the range of the data, flexible models with economic constraints can capture credit channels a straight line misses. No model, linear or not, reproduced the depth of either episode.

## 4. Translate the 2026 scenarios

Now train on all 1989–2025 quarters and feed in the Fed's paths.

In [ ]:
rows, paths = [], {}
for scenario, X in {"Severely adverse": X_severe, "Baseline": X_base}.items():
    preds = {name: m.predict(X.values) for name, m in full.items()}
    paths[scenario] = preds
    A.path_figure(preds, X.index, title=f"US bank stocks under the Fed's 2026 {scenario.lower()} scenario").show()
    for name, p in preds.items():
        level = A.cumulative(p)
        rows.append({"Scenario": scenario, "Model": name, "First quarter (%)": p[0], "Worst quarter (%)": p.min(),
                     "Lowest point (%)": level.min() - 100, "After 13 quarters (%)": level[-1] - 100})
translation = pd.DataFrame(rows).set_index(["Scenario", "Model"])
display(translation.round(1))

In the first quarter, the quarter that leaves history, the models disagree by a factor of two: linear regression says bank stocks fall 53.7%, boosting 29.9% and monotone boosting 26.1%. The trees stop at the worst outcomes they were trained on; the line keeps going. Over the whole scenario the lowest points are closer, from −63.1% to −72.7%, because the trees keep losing in later quarters that sit inside history.

Even the baseline, which never leaves history, divides the models: after 13 quarters they range from +8.2% to +40.2%. With 148 quarters of data, small differences in each quarter compound.

## 5. How unusual is the scenario?

One number per quarter: the Mahalanobis distance of the six inputs from their 1989–2025 average, the same plausibility yardstick as Lab 10b.

In [ ]:
mu, S = X_hist.mean().values, X_hist.cov().values
d_hist = pd.Series(A.mahalanobis_rows(X_hist.values, mu, S), index=X_hist.index)
d_severe = pd.Series(A.mahalanobis_rows(X_severe.values, mu, S), index=X_severe.index)
display(pd.concat({"Most unusual historical quarters": d_hist.nlargest(4),
                   "Severely adverse, first four quarters": d_severe.iloc[:4]}).round(1).to_frame("Distance"))

The scenario's first quarter sits at distance 9.2. Only one of the 148 historical quarters was further out: 2020Q2 (10.6). The worst quarter of the financial crisis, 2008Q4, was at 8.0. The Fed has written a quarter that is jointly more extreme than any in the global financial crisis, which is exactly the job of a severe scenario, and exactly where a model trained on history stops being informative.

## What to notice

1. **Read the scenario before translating it.** The 2026 severely adverse scenario leaves history in its first quarter, and it still takes short rates to 0.1%.
2. **Backtest the translation.** Given the true macro path, every model missed the depth of 2008 and 2020; monotone boosting came closest in 2008.
3. **Beyond history, the model is the assumption.** The first-quarter loss for bank stocks halves or doubles depending on whether the model can extrapolate. Report the range, not one number.

## Exercises

**Core 1.** Translate the scenario for another industry: change `target="Banks"` to `"RlEst"` (real estate) or `"Insur"` (insurers) and rerun. Which model disagrees most in the first quarter?

**Core 2.** Look at the baseline rows. Why do the models disagree after 13 quarters in a scenario that never leaves history? *Hint: compound a small difference in average quarterly return over 13 quarters.*

**Stretch 1.** Stressed calibration. Fit the linear model only on the quarters with the worst 25% of stock market returns, then translate the severely adverse scenario again. Is the first quarter more or less severe, and is that estimate more or less trustworthy?

**Stretch 2.** The Fed's own models translate the same scenario into bank capital: in the 2026 results the aggregate common equity tier 1 ratio fell by about 1.6 percentage points at its lowest. Explain how a fall of more than half in bank share prices can coexist with that. *Hint: market values against accounting capital, and what the stress test assumes about earnings and dividends.*